# Pair Visualization
<!-- structured-notebook -->
## Notebook Summary
Purpose: visualize the finalized cross-platform topic pairs for poster, paper, and internal review use.

Main steps:
- Load `stable_pairs_readable.csv` from `02_finalizing_pairs.ipynb`.
- Produce side-by-side per-pair diagnostics (top words, document counts, S values in both directions).
- Export figure assets for paper inclusion.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/topic_matchings/<pair>/stable_pairs_readable.csv` | Produced by `02_finalizing_pairs.ipynb` |
| Output | Figures for paper / poster (rendered in-notebook) | — |


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Visualize PubMed–Reddit topic pairs from a CSV.

Expected columns (auto-detected, case-insensitive):
- PubMed topic column  : any of ['pubmed_topic','pubmed','pm_topic','science_topic','left']
- Reddit topic column   : any of ['reddit_topic','reddit','rd_topic','social_topic','right']
- Optional weight/count : any of ['weight','count','n','freq','frequency','edge_weight']

Outputs:
- topic_pairs_heatmap.png (PubMed x Reddit grid of counts)
- topic_pairs_bipartite.png (network plot; thresholds optional)
- topic_pairs_sankey.html (if Plotly is available)

Usage:
    python visualize_topic_pairs.py --csv path/to/pairs.csv --out out_dir \
        --min_count 2 --top_pubmed 30 --top_reddit 30
"""

import argparse
import os
import sys
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cycler
from matplotlib.ticker import FuncFormatter
from textwrap import shorten

# -------- optional: network & sankey backends --------
try:
    import networkx as nx
    HAS_NX = True
except Exception:
    HAS_NX = False

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

# ----------------- Styling -----------------
def set_global_style():
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        try: plt.style.use("ggplot")
        except Exception: pass
    plt.rcParams.update({
        "figure.figsize": (9.5, 6),
        "figure.dpi": 140,
        "savefig.dpi": 220,
        "savefig.facecolor": "white",
        "savefig.bbox": "tight",
        "font.family": "DejaVu Sans",
        "axes.titlesize": 15,
        "axes.labelsize": 12.5,
        "xtick.labelsize": 10.5,
        "ytick.labelsize": 10.5,
        "legend.fontsize": 10.5,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titlepad": 10,
        "axes.prop_cycle": cycler("color", [
            "#3366CC","#DC3912","#FF9900","#109618","#990099",
            "#0099C6","#DD4477","#66AA00","#B82E2E","#316395"
        ]),
        "image.cmap": "viridis",
    })

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def savefig(path):
    try: plt.tight_layout()
    except Exception: pass
    plt.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.close()

# ----------------- Helpers -----------------
PM_CAND = ["pubmed_name", "pubmed_topic","pubmed","pm_topic","science_topic","left","source_topic","sci"]
RD_CAND = ["reddit_name", "reddit_topic","reddit","rd_topic","social_topic","right","target_topic","soc"]
WT_CAND = ["weight","count","n","freq","frequency","edge_weight","w"]

def pick_col(cols, candidates):
    lower = {c.lower(): c for c in cols}
    for name in candidates:
        if name in lower:
            return lower[name]
    # fuzzy match: startswith
    for c in cols:
        if any(c.lower().startswith(x.split("_")[0]) for x in candidates):
            return c
    return None

def coerce_topic(s: pd.Series) -> pd.Series:
    # normalize whitespace & trim
    return s.astype(str).fillna("").map(lambda x: re.sub(r"\s+", " ", x.strip()))

def load_pairs(csv_path: str):
    df = pd.read_csv(csv_path)
    cols = list(df.columns)
    pm_col = pick_col(cols, [c.lower() for c in PM_CAND]) or cols[0]
    rd_col = pick_col(cols, [c.lower() for c in RD_CAND]) or cols[1 if len(cols)>1 else 0]
    wt_col = pick_col(cols, [c.lower() for c in WT_CAND])

    df["_pm"] = coerce_topic(df[pm_col])
    df["_rd"] = coerce_topic(df[rd_col])

    if wt_col and wt_col in df.columns:
        df["_w"] = pd.to_numeric(df[wt_col], errors="coerce").fillna(1).astype(float)
    else:
        df["_w"] = 1.0

    # drop empties
    df = df[(df["_pm"] != "") & (df["_rd"] != "")]
    return df, pm_col, rd_col, wt_col

def top_filter(df, top_pubmed=None, top_reddit=None):
    res = df.copy()
    if top_pubmed:
        keep_pm = (res.groupby("_pm")["_w"].sum().sort_values(ascending=False)
                   .head(int(top_pubmed)).index)
        res = res[res["_pm"].isin(keep_pm)]
    if top_reddit:
        keep_rd = (res.groupby("_rd")["_w"].sum().sort_values(ascending=False)
                   .head(int(top_reddit)).index)
        res = res[res["_rd"].isin(keep_rd)]
    return res

# ----------------- Visuals -----------------
def plot_heatmap(pivot: pd.DataFrame, out_png: str, title="PubMed × Reddit topic pairs"):
    if pivot.empty:
        print("[info] Heatmap skipped (no data)."); return
    # Order rows/cols by totals for readability
    row_order = pivot.sum(axis=1).sort_values(ascending=True).index
    col_order = pivot.sum(axis=0).sort_values(ascending=True).index
    M = pivot.loc[row_order, col_order].values

    plt.figure(figsize=(max(8, 0.5*len(col_order)+3), max(6, 0.35*len(row_order)+2)))
    im = plt.imshow(M, aspect="auto", interpolation="nearest")
    plt.colorbar(label="Count / Weight")
    plt.title(title)
    # shorten labels to avoid overflows
    rlabels = [shorten(str(x), width=42, placeholder="…") for x in row_order]
    clabels = [shorten(str(x), width=36, placeholder="…") for x in col_order]
    plt.yticks(range(len(rlabels)), rlabels)
    plt.xticks(range(len(clabels)), clabels, rotation=45, ha="right")
    plt.xlabel("Reddit topics")
    plt.ylabel("PubMed topics")
    savefig(out_png)
    print("[ok] Heatmap →", out_png)

def plot_bipartite(df: pd.DataFrame, out_png: str, min_count=1, max_nodes=50, title="Bipartite topic graph"):
    if not HAS_NX:
        print("[warn] networkx not installed; skipping bipartite graph."); return
    agg = (df.groupby(["_pm","_rd"])["_w"].sum().reset_index())
    agg = agg[agg["_w"] >= float(min_count)]
    if agg.empty:
        print("[info] Bipartite skipped (no edges after filtering)."); return

    # Limit nodes by selecting strongest totals on each side
    pm_strength = agg.groupby("_pm")["_w"].sum().sort_values(ascending=False)
    rd_strength = agg.groupby("_rd")["_w"].sum().sort_values(ascending=False)
    pm_keep = set(pm_strength.head(max_nodes).index)
    rd_keep = set(rd_strength.head(max_nodes).index)
    agg = agg[agg["_pm"].isin(pm_keep) & agg["_rd"].isin(rd_keep)]
    if agg.empty:
        print("[info] Bipartite skipped (filtered out)."); return

    G = nx.Graph()
    # Add nodes with bipartite attribute
    for n in pm_keep:
        G.add_node(("pm", n), bipartite=0, side="PubMed")
    for n in rd_keep:
        G.add_node(("rd", n), bipartite=1, side="Reddit")
    # Add edges
    for _, r in agg.iterrows():
        G.add_edge(("pm", r["_pm"]), ("rd", r["_rd"]), weight=float(r["_w"]))

    # Layout: two columns (left/right) with vertical spacing based on degree
    left_nodes  = [n for n in G.nodes if n[0]=="pm"]
    right_nodes = [n for n in G.nodes if n[0]=="rd"]

    # Sort by strength to place important topics centered
    left_nodes  = sorted(left_nodes,  key=lambda n: pm_strength.get(n[1], 0), reverse=True)
    right_nodes = sorted(right_nodes, key=lambda n: rd_strength.get(n[1], 0), reverse=True)

    def stack_positions(nodes, x, gap=1.0):
        y = np.linspace(-1, 1, num=len(nodes)) if nodes else []
        return {n: (x, y[i] if len(y)>0 else 0.0) for i, n in enumerate(nodes)}

    pos = {}
    pos.update(stack_positions(left_nodes,  x=-1.0))
    pos.update(stack_positions(right_nodes, x= 1.0))

    # Edge widths scaled
    weights = np.array([G[u][v]["weight"] for u,v in G.edges()])
    if len(weights) == 0:
        print("[info] Bipartite has no edges."); return
    w_norm = 1.0 + 5.0 * (weights - weights.min()) / (weights.ptp() + 1e-9)

    plt.figure(figsize=(12, max(7, 0.28*(len(left_nodes)+len(right_nodes)))))
    # Draw edges first
    nx.draw_networkx_edges(G, pos, width=w_norm, alpha=0.35, edge_color="#555")
    # Node styling
    nx.draw_networkx_nodes(G, pos, nodelist=left_nodes,  node_color="#3366CC", node_size=260, alpha=0.9)
    nx.draw_networkx_nodes(G, pos, nodelist=right_nodes, node_color="#DC3912", node_size=260, alpha=0.9)

    # Labels (clipped)
    left_labels  = {n: shorten(n[1], width=28, placeholder="…") for n in left_nodes}
    right_labels = {n: shorten(n[1], width=28, placeholder="…") for n in right_nodes}
    nx.draw_networkx_labels(G, pos, labels={**left_labels, **right_labels}, font_size=9, font_color="white")

    # Legends
    from matplotlib.lines import Line2D
    legend_elems = [
        Line2D([0],[0], marker='o', color='w', label='PubMed', markerfacecolor="#3366CC", markersize=8),
        Line2D([0],[0], marker='o', color='w', label='Reddit', markerfacecolor="#DC3912", markersize=8),
        Line2D([0],[0], color="#555", lw=3, label='Edge ∝ count/weight', alpha=0.5),
    ]
    plt.legend(handles=legend_elems, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.06))

    plt.title(f"{title}\n(min_count={min_count}, nodes per side≤{max_nodes})", fontsize=13)
    plt.axis("off")
    savefig(out_png)
    print("[ok] Bipartite graph →", out_png)

def plot_sankey(df: pd.DataFrame, out_html: str, title="PubMed → Reddit topic flow"):
    if not HAS_PLOTLY:
        print("[warn] Plotly not installed; skipping Sankey. (pip install plotly)"); return

    agg = (df.groupby(["_pm","_rd"])["_w"].sum().reset_index())
    if agg.empty:
        print("[info] Sankey skipped (no data)."); return

    # Build label sets & index maps
    pm_labels = list(agg["_pm"].unique())
    rd_labels = list(agg["_rd"].unique())

    # Limit very large lists for readability (optional)
    MAX_L = 40
    if len(pm_labels) > MAX_L:
        pm_strength = agg.groupby("_pm")["_w"].sum().sort_values(ascending=False)
        pm_labels = list(pm_strength.head(MAX_L).index)
        agg = agg[agg["_pm"].isin(pm_labels)]
    if len(rd_labels) > MAX_L:
        rd_strength = agg.groupby("_rd")["_w"].sum().sort_values(ascending=False)
        rd_labels = list(rd_strength.head(MAX_L).index)
        agg = agg[agg["_rd"].isin(rd_labels)]

    labels = pm_labels + rd_labels
    pm_index = {t: i for i, t in enumerate(pm_labels)}
    rd_index = {t: i+len(pm_labels) for i, t in enumerate(rd_labels)}

    src = [pm_index[p] for p in agg["_pm"]]
    tgt = [rd_index[r] for r in agg["_rd"]]
    val = [float(w) for w in agg["_w"]]

    fig = go.Figure(data=[go.Sankey(
        node=dict(
            label=[shorten(s, width=42, placeholder="…") for s in labels],
            pad=15,
            thickness=12,
            line=dict(color="rgba(0,0,0,0.2)", width=0.5)
        ),
        link=dict(
            source=src,
            target=tgt,
            value=val,
        )
    )])
    fig.update_layout(title_text=title, font_size=11, width=1000, height=600)
    fig.write_html(out_html, include_plotlyjs="cdn")
    print("[ok] Sankey →", out_html)

# ----------------- Main -----------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--csv", required=True, help="Path to topic pairs CSV")
    parser.add_argument("--out", default="topic_pairs_viz", help="Output folder")
    parser.add_argument("--min_count", type=float, default=1.0, help="Min edge weight for bipartite")
    parser.add_argument("--top_pubmed", type=int, default=None, help="Limit to top-N PubMed topics by total weight (heatmap & network)")
    parser.add_argument("--top_reddit", type=int, default=None, help="Limit to top-N Reddit topics by total weight (heatmap & network)")
    args = parser.parse_args()

    set_global_style()
    ensure_dir(args.out)

    df, pm_col, rd_col, wt_col = load_pairs(args.csv)
    print(f"[loaded] {len(df):,} rows | PubMed col: '{pm_col}' | Reddit col: '{rd_col}' | Weight: '{wt_col or '1 per row'}'")

    # Optional top-k filtering for readability
    dff = top_filter(df, args.top_pubmed, args.top_reddit)

    # Pivot for heatmap (+ export)
    pivot = (dff.pivot_table(index="_pm", columns="_rd", values="_w", aggfunc="sum", fill_value=0))
    pivot_path = os.path.join(args.out, "topic_pairs_matrix.csv")
    pivot.to_csv(pivot_path)
    print("[ok] Matrix CSV →", pivot_path)
    plot_heatmap(pivot, os.path.join(args.out, "topic_pairs_heatmap.png"),
                 title="PubMed × Reddit topic pairs (count/weight)")

    # Bipartite network
    plot_bipartite(dff, os.path.join(args.out, "topic_pairs_bipartite.png"),
                   min_count=args.min_count, max_nodes=max(20, (args.top_pubmed or 40)))

    # Sankey (if plotly)
    plot_sankey(dff, os.path.join(args.out, "topic_pairs_sankey.html"),
                title="PubMed → Reddit topic flow")

if __name__ == "__main__":
    main()

---
<!-- nav-footer -->

← [02 finalizing pairs](../../../../../SocialMedia/Reddit/notebooks/BERTopic/04_topic_matching/02_finalizing_pairs.ipynb) | [Project Overview](../../../../../PROJECT_OVERVIEW.ipynb) | [04 topic doc exploration](../../../../../SocialMedia/Reddit/notebooks/BERTopic/04_topic_matching/04_topic_doc_exploration.ipynb) →
